# DALI 蛋白质结构比对工作流

这个 notebook 支持使用**在线 DALI 服务器**或**本地 DALI 安装**进行结构比对并保存结果。

**新特性：**
- ✨ 支持在线 DALI 服务器 (ekhidna2.biocenter.helsinki.fi)
- 🔄 自动模式：优先使用在线服务器，失败时自动回退到本地
- 📦 模块化：使用 `protflow.prediction.dali` 模块
- 🚀 批处理：高效处理多个结构
- 📊 结果分析：自动保存和可视化

## 目标

1. 使用**在线或本地 DALI** 对查询结构进行比对
2. 支持三种模式：
   - `online`: 使用在线 DALI 服务器
   - `local`: 使用本地 DALI 安装
   - `auto`: 自动选择（推荐）
3. 批处理多个 PDB 文件
4. 将结果转换成可视化或下游分析需要的格式

## 前提条件

- Python 3.10+（服务器已安装）
- **在线模式**: 需要网络访问 `ekhidna2.biocenter.helsinki.fi`
- **本地模式**: DALI 路径（例如 `dali.pl` 脚本）在 `PATH` 中或设置为绝对路径
- 结构文件统一放在 `data/structures/` 下，支持 `.pdb`、`.cif`、`.ent` 格式
- 已安装 `protflow` 包：`pip install -e .`

## DALI 数据库说明

### 在线模式
- 自动使用 DALI 在线服务器的最新数据库
- 支持的数据库：`pdb25`, `pdb50`, `pdb90`, `pdb100`
- 无需本地存储空间
- 网络连接要求稳定

### 本地模式
- 本地 DALI 安装默认依赖 `pdb100` 或 `pdb90` 数据库
- 请确保磁盘有 ≥50 GB 空间
- 推荐做法是在服务器上运行 `setup_dali_db.sh`（或参考官方 `fetch_dali.pl`）定期拉取最新 PDB 库
- 若仅做本地比对，可把 `DALI_DB_DIR` 环境变量指向挂载盘，然后在命令行参数中加入 `-databank <path>`

### 推荐使用在线模式
- 无需维护本地数据库
- 始终使用最新的 PDB 数据
- 节省磁盘空间

## 工作流程概览

1. 导入 `protflow.prediction.dali` 模块
2. 配置 DALI aligner（选择在线/本地/自动模式）
3. 枚举 query 结构文件
4. 对每个结构运行 DALI 并捕获结果
5. 解析输出，导出排名，必要时可视化

In [3]:
# DALI Structure Alignment - 内联版本# 简化的 DALI 比对工具，支持在线和本地模式import loggingimport osimport subprocessimport timefrom dataclasses import dataclassfrom pathlib import Pathfrom typing import Dict, List, Optional, Tuplefrom urllib.parse import urljoinimport pandas as pdimport requestslogging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")logger = logging.getLogger(__name__)@dataclassclass DaliResult:    """DALI 比对结果容器"""    query_name: str    target_pdb: str    rank: int    z_score: float    rmsd: float    lali: Optional[int] = None  # 比对长度    nres: Optional[int] = None  # 残基数    identity: Optional[float] = None  # 序列一致性 %        def to_dict(self) -> Dict:        """转换为字典"""        return {            'query': self.query_name,            'target_pdb': self.target_pdb,            'rank': self.rank,            'z_score': self.z_score,            'rmsd': self.rmsd,            'lali': self.lali,            'nres': self.nres,            'identity': self.identity,        }class DaliAligner:    """    DALI 结构比对工具，支持在线和本地模式        示例:        # 在线模式 (默认)        aligner = DaliAligner(mode='online')        results = aligner.align(Path('protein.pdb'))                # 本地模式        aligner = DaliAligner(mode='local', dali_cmd='/path/to/dali.pl')        results = aligner.align(Path('protein.pdb'))    """        ONLINE_SERVER = "https://ekhidna2.biocenter.helsinki.fi/dali/"    DEFAULT_TIMEOUT = 300  # 5分钟超时    POLL_INTERVAL = 10  # 每10秒检查一次状态        def __init__(        self,        mode: str = 'auto',        dali_cmd: Optional[Path] = None,        output_dir: Optional[Path] = None,        timeout: int = DEFAULT_TIMEOUT,        max_retries: int = 3,    ):        """        初始化 DALI 比对器                Args:            mode: 'online', 'local', 或 'auto' (先尝试在线，失败则本地)            dali_cmd: 本地 dali.pl 脚本路径 (用于本地模式)            output_dir: 输出目录 (默认: ./outputs/dali)            timeout: 在线查询超时时间（秒）            max_retries: 失败请求的最大重试次数        """        self.mode = mode.lower()        if self.mode not in ['online', 'local', 'auto']:            raise ValueError(f"无效模式: {mode}. 必须是 'online', 'local', 或 'auto'")                self.dali_cmd = Path(dali_cmd) if dali_cmd else self._find_dali_cmd()        self.output_dir = Path(output_dir) if output_dir else Path.cwd() / "outputs" / "dali"        self.output_dir.mkdir(parents=True, exist_ok=True)                self.timeout = timeout        self.max_retries = max_retries        def _find_dali_cmd(self) -> Optional[Path]:        """查找本地 DALI 命令"""        common_paths = [            Path("/usr/local/bin/dali.pl"),            Path("/opt/dali/dali.pl"),            Path.home() / "bin" / "dali.pl",        ]        for path in common_paths:            if path.exists():                return path        return None        def _check_local_dali(self) -> bool:        """检查本地 DALI 是否可用"""        if not self.dali_cmd or not self.dali_cmd.exists():            return False        try:            result = subprocess.run(                [str(self.dali_cmd), "--help"],                capture_output=True,                timeout=10,            )            return result.returncode == 0 or "dali" in result.stdout.decode().lower()        except (subprocess.TimeoutExpired, FileNotFoundError):            return False        def _check_online_availability(self) -> bool:        """检查在线 DALI 服务器是否可访问"""        try:            response = requests.get(self.ONLINE_SERVER, timeout=10)            return response.status_code == 200        except Exception as e:            logger.debug(f"在线 DALI 服务器不可访问: {e}")            return False        def align(        self,        query_structure: Path,        database: str = "pdb25",        output_name: Optional[str] = None,    ) -> List[DaliResult]:        """        对查询结构进行比对                Args:            query_structure: 查询 PDB 文件路径            database: 要搜索的数据库 (仅在线模式)            output_name: 输出名称 (默认: 查询文件名)                Returns:            DaliResult 对象列表        """        if not query_structure.exists():            raise FileNotFoundError(f"查询结构不存在: {query_structure}")                if output_name is None:            output_name = query_structure.stem                # 确定使用的模式        actual_mode = self._determine_mode()                # 执行比对        if actual_mode == 'online':            results = self._align_online(query_structure, database, output_name)        else:            results = self._align_local(query_structure, output_name)                # 保存结果        self._save_results(results, output_name)                return results        def _determine_mode(self) -> str:        """确定实际使用的模式"""        if self.mode == 'online':            if not self._check_online_availability():                raise RuntimeError("在线 DALI 服务器不可用")            return 'online'        elif self.mode == 'local':            if not self._check_local_dali():                raise RuntimeError("本地 DALI 不可用")            return 'local'        else:  # auto 模式            if self._check_online_availability():                logger.info("使用在线 DALI 服务器")                return 'online'            elif self._check_local_dali():                logger.info("回退到本地 DALI")                return 'local'            else:                raise RuntimeError("在线和本地 DALI 都不可用")        def _align_online(        self,        query_structure: Path,        database: str,        output_name: str,    ) -> List[DaliResult]:        """使用在线 DALI 服务器执行比对"""        logger.info(f"提交 {query_structure.name} 到在线 DALI 服务器...")                # 提交任务        job_id = self._submit_online_job(query_structure, database)                # 轮询结果        results_data = self._poll_online_results(job_id)                # 解析结果        results = self._parse_online_results(results_data, output_name)                logger.info(f"找到 {len(results)} 个比对结果")        return results        def _submit_online_job(self, query_structure: Path, database: str) -> str:        """提交在线 DALI 任务"""        submit_url = urljoin(self.ONLINE_SERVER, "api/submit")                with open(query_structure, 'rb') as f:            files = {'pdbfile': (query_structure.name, f, 'application/octet-stream')}            data = {'database': database}                        for attempt in range(self.max_retries):                try:                    response = requests.post(submit_url, files=files, data=data, timeout=30)                    response.raise_for_status()                    result = response.json()                                        if 'job_id' not in result:                        raise RuntimeError("服务器响应缺少 'job_id' 字段")                                        return result['job_id']                                    except requests.RequestException as e:                    if attempt == self.max_retries - 1:                        raise RuntimeError(f"提交 DALI 任务失败: {e}")                    logger.warning(f"尝试 {attempt + 1} 失败，重试中...")                    time.sleep(5)        def _poll_online_results(self, job_id: str) -> Dict:        """轮询 DALI 服务器等待任务完成"""        status_url = urljoin(self.ONLINE_SERVER, f"api/status/{job_id}")        result_url = urljoin(self.ONLINE_SERVER, f"api/result/{job_id}")                start_time = time.time()        while time.time() - start_time < self.timeout:            try:                response = requests.get(status_url, timeout=10)                response.raise_for_status()                status = response.json()                                if status.get('status') == 'completed':                    result_response = requests.get(result_url, timeout=30)                    result_response.raise_for_status()                    return result_response.json()                                    elif status.get('status') == 'failed':                    raise RuntimeError(f"DALI 任务失败: {status.get('error', '未知错误')}")                                time.sleep(self.POLL_INTERVAL)            except requests.RequestException as e:                logger.warning(f"轮询状态错误: {e}")                time.sleep(self.POLL_INTERVAL)                raise TimeoutError(f"DALI 任务 {job_id} 在 {self.timeout}秒内未完成")        def _parse_online_results(self, results_data: Dict, query_name: str) -> List[DaliResult]:        """解析在线 DALI 服务器的结果"""        results = []                for idx, hit in enumerate(results_data.get('hits', []), start=1):            result = DaliResult(                query_name=query_name,                target_pdb=hit.get('pdbid', ''),                rank=idx,                z_score=float(hit.get('z', 0.0)),                rmsd=float(hit.get('rmsd', 0.0)),                lali=int(hit.get('lali', 0)) if hit.get('lali') else None,                nres=int(hit.get('nres', 0)) if hit.get('nres') else None,                identity=float(hit.get('id', 0.0)) if hit.get('id') else None,            )            results.append(result)                return sorted(results, key=lambda x: x.z_score, reverse=True)        def _align_local(self, query_structure: Path, output_name: str) -> List[DaliResult]:        """使用本地 DALI 安装执行比对"""        logger.info(f"为 {query_structure.name} 运行本地 DALI...")                query_output_dir = self.output_dir / output_name        query_output_dir.mkdir(parents=True, exist_ok=True)                cmd = [            str(self.dali_cmd),            "-query", str(query_structure),            "-hera", str(query_output_dir),        ]                try:            process = subprocess.run(cmd, capture_output=True, text=True, timeout=self.timeout)                        if process.returncode != 0:                raise RuntimeError(f"DALI 失败，代码 {process.returncode}: {process.stderr}")                        # 解析结果            log_file = query_output_dir / "dali.log"            if not log_file.exists():                for pattern in ["*.log", "*-dali.txt"]:                    logs = list(query_output_dir.glob(pattern))                    if logs:                        log_file = logs[0]                        break                        if not log_file.exists():                raise FileNotFoundError(f"在 {query_output_dir} 中未找到 DALI 日志文件")                        results = self._parse_dali_log(log_file, output_name)            logger.info(f"为 {query_structure.name} 找到 {len(results)} 个比对")            return results                    except subprocess.TimeoutExpired:            raise TimeoutError(f"本地 DALI 在 {self.timeout}秒后超时")        def _parse_dali_log(self, log_path: Path, query_name: str) -> List[DaliResult]:        """解析 DALI 日志文件"""        results = []                with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:            for line in f:                line = line.strip()                                if not line or line.startswith('#'):                    continue                                parts = line.split()                if len(parts) < 4:                    continue                                try:                    result = DaliResult(                        query_name=query_name,                        rank=int(parts[0]),                        target_pdb=parts[1],                        z_score=float(parts[2]) if len(parts) > 2 else 0.0,                        rmsd=float(parts[3]) if len(parts) > 3 else 0.0,                        lali=int(parts[4]) if len(parts) > 4 else None,                        nres=int(parts[5]) if len(parts) > 5 else None,                        identity=float(parts[6]) if len(parts) > 6 else None,                    )                    results.append(result)                except (ValueError, IndexError):                    continue                return sorted(results, key=lambda x: x.z_score, reverse=True)        def _save_results(self, results: List[DaliResult], output_name: str):        """保存结果到 CSV 文件"""        if not results:            logger.warning("没有结果可保存")            return                output_path = self.output_dir / f"{output_name}_results.csv"                df = pd.DataFrame([r.to_dict() for r in results])        df.to_csv(output_path, index=False)        logger.info(f"结果已保存到 {output_path}")        def align_batch(        self,        query_structures: List[Path],        database: str = "pdb25",    ) -> List[Tuple[str, List[DaliResult]]]:        """批量比对多个结构"""        all_results = []                for query in query_structures:            try:                results = self.align(query, database=database)                all_results.append((query.stem, results))            except Exception as e:                logger.error(f"处理 {query.name} 失败: {e}")                return all_results        def summarize_results(        self,        results_list: List[Tuple[str, List[DaliResult]]],        top_n: int = 10,    ) -> pd.DataFrame:        """汇总多个查询的结果"""        all_data = []        for query_name, results in results_list:            for result in results[:top_n]:                all_data.append(result.to_dict())                if not all_data:            return pd.DataFrame()                df = pd.DataFrame(all_data)        df = df.sort_values(by='z_score', ascending=False)                return dfdef batch_align(    structures_dir: Path,    pattern: str = "*.pdb",    mode: str = 'auto',    output_dir: Optional[Path] = None,) -> List[Tuple[str, List[DaliResult]]]:    """便捷函数：比对目录中的所有结构"""    if not structures_dir.exists():        raise FileNotFoundError(f"目录不存在: {structures_dir}")        structures = sorted(structures_dir.glob(pattern))    if not structures:        logger.warning(f"在 {structures_dir} 中未找到匹配 {pattern} 的结构")        return []        logger.info(f"找到 {len(structures)} 个结构文件")        aligner = DaliAligner(mode=mode, output_dir=output_dir)    return aligner.align_batch(structures)print("✓ DALI 比对工具已加载")

/opt/anaconda3/envs/ESM3/lib/python3.13/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [5]:
ROOT = Path().resolve()
STRUCTURES_DIR = ROOT / "data" / "structures"
OUTPUT_BASE = ROOT / "outputs" / "dali"
ESM3_PRED_DIR = ROOT / "outputs" / "esm3" / "predictions"

# DALI Configuration
DALI_MODE = 'auto'  # 'online', 'local', or 'auto'
DALI_DATABASE = 'pdb25'  # For online mode: pdb25, pdb50, pdb90, pdb100
DALI_CMD = None  # Path to dali.pl for local mode (None = auto-detect)

if not STRUCTURES_DIR.exists():
    STRUCTURES_DIR.mkdir(parents=True, exist_ok=True)
    logging.warning(f"{STRUCTURES_DIR} 已创建，请放入结构文件。")

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

print(f"配置：")
print(f"  模式: {DALI_MODE}")
print(f"  数据库: {DALI_DATABASE}")
print(f"  结构目录: {STRUCTURES_DIR}")
print(f"  输出目录: {OUTPUT_BASE}")

配置：
  模式: auto
  数据库: pdb25
  结构目录: /Users/asagiri/PycharmProjects/ProtFlow/notebooks/tools/data/structures
  输出目录: /Users/asagiri/PycharmProjects/ProtFlow/notebooks/tools/outputs/dali


## 初始化 DALI Aligner

创建 DALI aligner 实例。根据 `DALI_MODE` 配置选择工作模式：
- `online`: 使用在线 DALI 服务器
- `local`: 使用本地 DALI 安装
- `auto`: 自动选择（优先在线，失败时回退到本地）

In [6]:
# Initialize DALI aligner
aligner = DaliAligner(
    mode=DALI_MODE,
    dali_cmd=DALI_CMD,
    output_dir=OUTPUT_BASE,
    timeout=300,  # 5 minutes for online queries
)

print(f"✓ DALI Aligner 已初始化")
print(f"  实际使用模式将在运行时确定")

INFO: DALI aligner initialized in auto mode


✓ DALI Aligner 已初始化
  实际使用模式将在运行时确定


## 查询结构准备

列出所有待比对的 PDB/CIF/ENT 文件，确保命名一致，便于批处理。

## ESM3 结果整合（萜合酶探索）

为了锁定潜在萜类合酶，可以把 ESM3 Workflow 的预测结构（PDB）自动同步到 `data/structures/`，然后交给 DALI 做结构相似性搜索。
- 默认假设预测结果保存在 `outputs/esm3/predictions/`，文件名和 `target_id` 一致
- 同步时会为每个文件加上 `esm3_` 前缀，避免覆盖手工准备的结构
- 结合 Z-score、高度保守的活性位点和注释，可以快速筛查疑似萜合酶

In [7]:
def sync_esm3_predictions(source_dir: Path = ESM3_PRED_DIR, target_dir: Path = STRUCTURES_DIR) -> list[Path]:
    synced = []
    if not source_dir.exists():
        logging.info("ESM3 预测目录 %s 不存在，跳过同步。", source_dir)
        return synced
    target_dir.mkdir(parents=True, exist_ok=True)
    for ext in ("*.pdb", "*.cif"):
        for structure in sorted(source_dir.glob(ext)):
            dest = target_dir / f"esm3_{structure.name}"
            if not dest.exists() or structure.stat().st_mtime > dest.stat().st_mtime:
                shutil.copy2(structure, dest)
                logging.info("同步 %s -> %s", structure.name, dest.name)
            synced.append(dest)
    return synced

In [8]:
synced_esm3 = sync_esm3_predictions()
query_patterns = ["*.pdb", "*.cif", "*.ent"]
queries: list[Path] = []
for pattern in query_patterns:
    queries.extend(sorted(STRUCTURES_DIR.glob(pattern)))
if synced_esm3:
    logging.info("已加入 %d 个 ESM3 预测结构用于 DALI。", len(synced_esm3))
# 去重保持顺序
unique: list[Path] = []
seen: set[Path] = set()
for path in queries:
    if path not in seen:
        unique.append(path)
        seen.add(path)
queries = unique
if not queries:
    logging.warning("在 %s 中没有发现 PDB/CIF 文件，请先放入结构。", STRUCTURES_DIR)
else:
    print(
        "找到 %d 个结构文件，前 5 个：" % len(queries),
        *[q.name for q in queries[:5]],
        sep="\n",
    )

INFO: ESM3 预测目录 /Users/asagiri/PycharmProjects/ProtFlow/notebooks/tools/outputs/esm3/predictions 不存在，跳过同步。


找到 5370 个结构文件，前 5 个：
AIOBBLFC_00001.pdb
AIOBBLFC_00002.pdb
AIOBBLFC_00003.pdb
AIOBBLFC_00009.pdb
AIOBBLFC_00010.pdb


## DALI 结构比对

使用新的 `DaliAligner` 类进行结构比对。支持：
- 单个结构比对
- 批量处理
- 在线和本地模式自动切换

In [9]:
# 示例：比对单个结构
if queries:
    sample_query = queries[0]
    print(f"比对示例: {sample_query.name}")
    
    results = aligner.align(
        query_structure=sample_query,
        database=DALI_DATABASE,
    )
    
    print(f"\n找到 {len(results)} 个比对结果")
    if results:
        print(f"\n前 5 个结果：")
        for r in results[:5]:
            print(f"  {r.rank}. {r.target_pdb:8s} Z={r.z_score:6.2f}  RMSD={r.rmsd:5.2f}")
else:
    print("未找到结构文件，请先放入 data/structures/")

比对示例: AIOBBLFC_00001.pdb


RuntimeError: Neither online nor local DALI available

## 批量处理和结果汇总

使用 `align_batch` 方法批量处理所有结构，并生成汇总表格。

In [ ]:
# 批量比对所有结构
if queries:
    print(f"批量处理 {len(queries)} 个结构...")
    
    batch_results = aligner.align_batch(
        query_structures=queries,
        database=DALI_DATABASE,
    )
    
    # 创建汇总表
    summary_df = aligner.summarize_results(batch_results, top_n=10)
    
    if summary_df is not None and not summary_df.empty:
        print(f"\n汇总：找到 {len(summary_df)} 个总比对结果")
        display(summary_df.head(20))
        
        # 保存汇总
        summary_path = OUTPUT_BASE / "dali_batch_summary.csv"
        summary_df.to_csv(summary_path, index=False)
        logging.info(f"批量汇总已保存到 {summary_path}")
    else:
        print("没有结果可汇总")
else:
    print("没有结构文件可处理")

## 示例执行

若已经准备好了结构文件，可先跑一条记录“绿灯”全链路。

In [ ]:
if queries:
    sample_query = queries[0]
    target = OUTPUT_BASE / sample_query.stem
    log_path = run_local_dali(sample_query, target)
    print("示例日志：", log_path)
else:
    print("未找到结构文件，请先放入 data/structures。")


## 下一步

### 可视化和分析
- 如果需要可视化，可将 `results` 传给 `nglview`/`py3Dmol`
- 可以根据 Z-score 筛选高相似度结构
- 结合序列比对和功能注释进行综合分析

### 自动化和集成
- 可将批处理包装进 `papermill`/`nbconvert` 生成报告
- 集成到 Prokka → ESM3 → DALI 工作流
- 添加到 CLI 脚本中实现命令行调用

### 模式切换
如果在线模式不可用，只需修改配置：
```python
DALI_MODE = 'local'  # 切换到本地模式
DALI_CMD = Path('/path/to/dali.pl')  # 指定 DALI 命令路径
```

### 高级用法
```python
# 使用 Python API 直接调用
from protflow.prediction.dali import run_dali_alignment

results = run_dali_alignment(
    query_structure=Path('protein.pdb'),
    mode='online',
    database='pdb25',
)
```

*注：此单元格已移除，不再需要添加 protflow 到路径*